In [14]:
# ============================================
# SMART RAINWATER SYSTEM – DATA PREPARATION
# Initial Notebook Cell
# ============================================

import pandas as pd
import numpy as np

# -------------------------------
# 1. LOAD DATASETS (EDIT PATHS)
# -------------------------------

# Rainfall datasets
rainfall_ts_path = r"E:\Documents\SEM-8\SDP\dataset\rainfall in india 1901-2015.csv"
district_rainfall_path = r"E:\Documents\SEM-8\SDP\dataset\district wise rainfall normal.csv"

# Smart rainwater harvesting dataset
smart_harvest_path = r"E:\Documents\SEM-8\SDP\dataset\Smart_Rainwater_Harvesting_Dataset.csv"

# Water quality / potability dataset
water_quality_path = r"E:\Documents\SEM-8\SDP\dataset\water_potability.csv"

# Load CSVs
rainfall_ts = pd.read_csv(rainfall_ts_path)
district_rainfall = pd.read_csv(district_rainfall_path)
harvest_df = pd.read_csv(smart_harvest_path)
water_df = pd.read_csv(water_quality_path)

print("Datasets loaded successfully")


Datasets loaded successfully


In [16]:
# -------------------------------
# 2. STANDARDIZE & SELECT FEATURES
# -------------------------------

# Clean column names
for df in [rainfall_ts, district_rainfall, harvest_df, water_df]:
    df.columns = df.columns.str.lower().str.strip()

# Select useful columns (adjust if names differ)
rainfall_ts = rainfall_ts[['state', 'year', 'annual']]
district_rainfall = district_rainfall[['state', 'district', 'annual']]
harvest_df = harvest_df[['rainfall_mm', 'rainfall_mm', 'roof_area(sqm)', 'water_collected_liters', 'water_used_liters']].rename(columns={'roof_area(sqm)': 'roof_area_m2'})
water_df = water_df[['ph','hardness', 'solids', 'turbidity', 'potability']].rename(columns={'solids': 'tds'})

print("Feature selection complete")


Feature selection complete


In [17]:
# ----------------------------------
# 3. CREATE LOCATION MASTER TABLE
# ----------------------------------

# Sample realistic Indian locations
locations = [
    ("Hyderabad", "Hyderabad", "Telangana"),
    ("Vijayawada", "Krishna", "Andhra Pradesh"),
    ("Bengaluru", "Bangalore Urban", "Karnataka"),
    ("Chennai", "Chennai", "Tamil Nadu"),
    ("Mumbai", "Mumbai", "Maharashtra"),
    ("Pune", "Pune", "Maharashtra"),
    ("Delhi", "New Delhi", "Delhi"),
    ("Kolkata", "Kolkata", "West Bengal"),
    ("Ahmedabad", "Ahmedabad", "Gujarat"),
    ("Jaipur", "Jaipur", "Rajasthan")
]

location_df = pd.DataFrame(
    locations, columns=["city", "district", "state"]
)

print("Location master created")


Location master created


In [18]:
# ----------------------------------
# 4. SYNTHETIC RAIN PARAMETER GENERATION
# ----------------------------------

np.random.seed(42)

n = len(location_df)

synthetic_rain = pd.DataFrame({
    "rainfall_intensity_mm_hr": np.random.uniform(2, 60, n),
    "raindrop_size_mm": np.random.uniform(0.5, 5.0, n),
    "rainfall_type": np.random.choice(
        ["Drizzle", "Moderate", "Heavy", "Storm"], n
    ),
    "roof_area_m2": np.random.uniform(50, 200, n)
})

location_rain_df = pd.concat([location_df, synthetic_rain], axis=1)

print("Synthetic rainfall parameters generated")


Synthetic rainfall parameters generated


In [19]:
# ----------------------------------
# 5. SYNTHETIC WATER QUALITY MAPPING
# ----------------------------------

water_synth = water_df.sample(n, replace=True).reset_index(drop=True)

final_df = pd.concat([location_rain_df, water_synth], axis=1)

# Rename for clarity
final_df.rename(columns={
    "solids": "tds",
    "potability": "harvestable_label"
}, inplace=True)

print("Water quality features merged")


Water quality features merged


In [20]:
# ----------------------------------
# 6. DERIVED TARGET FEATURES
# ----------------------------------

# Harvestable quantity (liters)
final_df["harvest_quantity_liters"] = (
    final_df["rainfall_intensity_mm_hr"]
    * final_df["roof_area_m2"]
    * 0.85
)

# Harvestable yes/no
final_df["harvestable"] = np.where(
    (final_df["ph"].between(6.5, 8.5)) &
    (final_df["turbidity"] < 5),
    1, 0
)

# Rain kinetic energy estimation (simplified)
# KE = 0.5 * m * v^2 (proxy)
final_df["rain_energy_joules"] = (
    0.5
    * (final_df["raindrop_size_mm"] / 1000)
    * (final_df["rainfall_intensity_mm_hr"] ** 2)
)

print("Harvest & energy features added")


Harvest & energy features added


In [21]:
# ----------------------------------
# 7. FINAL DATASET
# ----------------------------------

final_columns = [
    "city", "district", "state",
    "rainfall_intensity_mm_hr",
    "raindrop_size_mm",
    "rainfall_type",
    "roof_area_m2",
    "ph", "tds", "turbidity",
    "harvestable",
    "harvest_quantity_liters",
    "rain_energy_joules"
]

final_dataset = final_df[final_columns]

final_dataset.head()


,city,district,state,rainfall_intensity_mm_hr,raindrop_size_mm,rainfall_type,roof_area_m2,ph,tds,turbidity,harvestable,harvest_quantity_liters,rain_energy_joules
0,Hyderabad,Hyderabad,Telangana,23.723327,0.592630,Heavy,167.776394,7.810145,20418.889926,4.033774,1,3383.182108,0.166765
1,Vijayawada,Krishna,Andhra Pradesh,57.141430,4.864594,Heavy,79.951067,6.420640,26034.528198,3.139940,0,3883.240554,7.941798
2,Bengaluru,Bangalore Urban,Karnataka,44.455649,4.245992,Moderate,127.135166,8.801846,13121.171943,4.031496,0,4804.094818,4.195687
3,Chennai,Chennai,Tamil Nadu,36.722192,1.455526,Storm,138.862185,3.854618,19639.796002,4.290435,0,4334.425266,0.981403
4,Mumbai,Mumbai,Maharashtra,11.049081,1.318212,Storm,56.967562,11.496702,14158.157283,3.132253,0,535.023332,0.080465


In [22]:
# ----------------------------------
# 8. SAVE FINAL DATASET
# ----------------------------------

final_dataset_path = r"E:\Documents\SEM-8\SDP\dataset\final_smart_rainwater_dataset.csv"

final_dataset.to_csv(final_dataset_path, index=False)

print(f"Final dataset saved at: {final_dataset_path}")


Final dataset saved at: E:\Documents\SEM-8\SDP\dataset\final_smart_rainwater_dataset.csv


In [23]:
# ----------------------------------
# 9. DATA AUGMENTATION (10,000+ ROWS)
# ----------------------------------

np.random.seed(42)

BASE_DF = final_dataset.copy()

TARGET_ROWS = 10000
augmented_rows = []

while len(augmented_rows) < TARGET_ROWS:

    sample = BASE_DF.sample(1).iloc[0]

    # -----------------------------
    # Rainfall augmentation
    # -----------------------------
    rainfall_intensity = np.clip(
        sample["rainfall_intensity_mm_hr"] * np.random.uniform(0.7, 1.3),
        1, 120
    )

    raindrop_size = np.clip(
        sample["raindrop_size_mm"] * np.random.uniform(0.8, 1.2),
        0.3, 6.0
    )

    roof_area = np.clip(
        sample["roof_area_m2"] * np.random.uniform(0.85, 1.15),
        30, 300
    )

    # -----------------------------
    # Water quality augmentation
    # -----------------------------
    ph = np.clip(
        sample["ph"] + np.random.normal(0, 0.25),
        5.5, 9.5
    )

    tds = np.clip(
        sample["tds"] * np.random.uniform(0.85, 1.2),
        50, 2500
    )

    turbidity = np.clip(
        sample["turbidity"] * np.random.uniform(0.8, 1.3),
        0.1, 15
    )

    # -----------------------------
    # Recalculate targets
    # -----------------------------
    harvestable = int((6.5 <= ph <= 8.5) and (turbidity < 5))

    harvest_quantity = (
        rainfall_intensity * roof_area * 0.85
        if harvestable else 0
    )

    rain_energy = (
        0.5
        * (raindrop_size / 1000)
        * (rainfall_intensity ** 2)
    )

    augmented_rows.append({
        "city": sample["city"],
        "district": sample["district"],
        "state": sample["state"],
        "rainfall_intensity_mm_hr": rainfall_intensity,
        "raindrop_size_mm": raindrop_size,
        "rainfall_type": sample["rainfall_type"],
        "roof_area_m2": roof_area,
        "ph": ph,
        "tds": tds,
        "turbidity": turbidity,
        "harvestable": harvestable,
        "harvest_quantity_liters": harvest_quantity,
        "rain_energy_joules": rain_energy
    })

augmented_df = pd.DataFrame(augmented_rows)

print("Augmented dataset shape:", augmented_df.shape)


Augmented dataset shape: (10000, 13)


In [25]:
# ----------------------------------
# 10. SAVE AUGMENTED DATASET
# ----------------------------------

augmented_dataset_path = r"E:\Documents\SEM-8\SDP\dataset\final_augmented_rainwater_dataset.csv"

augmented_df.to_csv(augmented_dataset_path, index=False)

print("Augmented dataset saved:", augmented_dataset_path)


Augmented dataset saved: E:\Documents\SEM-8\SDP\dataset\final_augmented_rainwater_dataset.csv
